# LeetCode 366 — Find Leaves of Binary Tree bằng Python

Link đề: https://leetcode.com/problems/find-leaves-of-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree, Tree DP

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findLeaves(self, root: Optional[TreeNode]) -> List[List[int]]:
        ...
```

Ghi chú: đây là bài **Premium** nên đề được viết theo hiểu biết về bài; hãy đối chiếu lại với đề gốc nếu bạn có tài khoản Premium. LeetCode đã cung cấp sẵn `TreeNode`; trong notebook này ta **tự dựng** `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (`[1,2,3,4,5]`, `None` là nút trống) để chạy thử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lặp lại "**thu thập mọi lá hiện tại → xóa chúng**" cho đến khi cây rỗng; kết quả là danh sách các **đợt** lá (mỗi đợt một list).
2. Nắm các công cụ Python: lớp `TreeNode`, dựng cây từ list kiểu LeetCode bằng `deque`, **đệ quy hậu thứ tự (postorder)**, `list` các list (`ans[h].append(...)`), và ngăn xếp tường minh cho DFS không đệ quy.
3. Đi từ cách **mô phỏng thật sự việc xóa lá từng đợt** → **tính "chiều cao" của mỗi nút bằng một lượt DFS** → **bóc lớp bằng đếm số con (kiểu sắp xếp topo)** → **DFS hậu thứ tự bằng ngăn xếp**.
4. Hiểu mấu chốt: **một nút bị xóa ở đợt `h`**, trong đó `h` là **chiều cao** của nó (lá có `h = 0`, nút khác có `h = 1 + max(h_trái, h_phải)`), vì nút chỉ trở thành lá **sau khi cả hai con đã bị xóa**, tức là sau đợt muộn hơn của hai con.
5. Tránh các lỗi kinh điển: nhầm **chiều cao** với **độ sâu**, dùng `min` thay `max`, quên tạo list mới cho đợt mới (`IndexError`), mô phỏng quên cắt liên kết cha–con (lặp vô hạn).

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân. Hãy thu thập các nút của cây như sau:

- Thu thập **tất cả các nút lá**.
- **Xóa** tất cả các nút lá đó.
- Lặp lại cho đến khi cây **rỗng**.

Trả về danh sách các đợt (mỗi đợt là list các giá trị lá được thu thập ở lượt đó), theo thứ tự các lượt.

**Ví dụ 1**

- Input: `root = [1,2,3,4,5]`
- Output: `[[4,5,3],[2],[1]]`
- Giải thích: lượt 1 các lá là `4, 5, 3`; xóa chúng còn cây `1 → 2`; lượt 2 lá là `2`; lượt 3 còn `1`. Với **mỗi đợt**, **thứ tự các nút không quan trọng**: `[[3,5,4],[2],[1]]` và `[[3,4,5],[2],[1]]` cũng được chấp nhận.

**Ví dụ 2**

- Input: `root = [1]`
- Output: `[[1]]`

**Ràng buộc:** số nút trong cây trong khoảng `[1, 100]`; `-100 <= Node.val <= 100`.

(Cây được cho dưới dạng danh sách theo thứ tự từng tầng, `null` là nút trống — hình minh họa của đề được thay bằng chữ ở đây.)

In [1]:
import random
from collections import deque
from typing import List, Optional

examples_lists = [
    ([1, 2, 3, 4, 5], [[4, 5, 3], [2], [1]]),
    ([1], [[1]]),
]
for lst, expected in examples_lists:
    print(lst, "-> kỳ vọng", expected)

[1, 2, 3, 4, 5] -> kỳ vọng [[4, 5, 3], [2], [1]]
[1] -> kỳ vọng [[1]]


## 3. Kiến thức nền

### 3.1. Nút cây `TreeNode`

Mỗi nút của cây nhị phân có ba thứ: `val` (giá trị), `left` (con trái) và `right` (con phải); con trống là `None`. **Nút lá** là nút có **cả hai con đều `None`**. Ta dựng lớp này giống LeetCode (LeetCode đã có sẵn, **không** khai báo lại khi nộp).

In [2]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


# cây:   1
#       / \
#      2   3
leaf2 = TreeNode(2)
leaf3 = TreeNode(3)
root = TreeNode(1, leaf2, leaf3)
print(root.val, root.left.val, root.right.val)           # 1 2 3
print(leaf2.left is None and leaf2.right is None)        # True: leaf2 là lá
print(root.left is None or root.right is None)           # False: root có đủ hai con (không phải lá)

1 2 3
True
False


### 3.2. Dựng cây từ list kiểu LeetCode: `build_tree`

LeetCode cho cây dưới dạng list theo **thứ tự từng tầng** (level order), `None` đánh dấu chỗ trống. Ví dụ `[1,2,3,None,4]`: gốc `1`; con của `1` là `2` và `3`; `2` không có con trái (`None`) nhưng có con phải `4`. Cách dựng: dùng **hàng đợi** các nút đang chờ nhận con; lấy một nút ra, gán 2 giá trị kế tiếp của list làm con trái/phải (nếu không `None` thì tạo nút và đưa vào hàng đợi).

In [3]:
def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (level order, None = nút trống)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (bỏ các None thừa ở cuối)."""
    if root is None:
        return []
    out = []
    q = deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            q.append(node.left)
            q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


t = build_tree([1, 2, 3, None, 4])
print(t.val, t.left.val, t.right.val, t.left.left, t.left.right.val)    # 1 2 3 None 4
print(tree_to_list(t))                                                   # [1, 2, 3, None, 4]
print(tree_to_list(build_tree([1, 2, 3, 4, 5])))                         # [1, 2, 3, 4, 5]

1 2 3 None 4
[1, 2, 3, None, 4]
[1, 2, 3, 4, 5]


### 3.3. Hình dung cây bằng chữ

Để "nhìn" cây, ta in nghiêng (xoay 90 độ): con phải ở trên, con trái ở dưới, thụt lề theo độ sâu. Đây chỉ là công cụ hiển thị.

In [4]:
def show_tree(node, depth=0):
    if node is None:
        return
    show_tree(node.right, depth + 1)
    print("    " * depth + str(node.val))
    show_tree(node.left, depth + 1)


show_tree(build_tree([1, 2, 3, 4, 5]))

    3
1
        5
    2
        4


### 3.4. Đệ quy và thứ tự duyệt: **postorder** (hậu thứ tự)

Duyệt cây bằng đệ quy có ba kiểu: **preorder** (nút, trái, phải), **inorder** (trái, nút, phải), **postorder** (**trái, phải, rồi nút**). **Postorder** quan trọng ở bài này: khi ta "đến" nút, **cả hai cây con đã được xử lý xong**, nên có thể dùng kết quả của chúng (ví dụ chiều cao) để tính cho nút hiện tại.

In [5]:
def postorder_values(node):
    if node is None:
        return []
    return postorder_values(node.left) + postorder_values(node.right) + [node.val]


print(postorder_values(build_tree([1, 2, 3, 4, 5])))      # [4, 5, 2, 3, 1]: con trước, cha sau

[4, 5, 2, 3, 1]


### 3.5. Chiều cao của nút (height) so với độ sâu (depth)

- **Độ sâu** của nút = số cạnh **từ gốc xuống nút** (gốc có độ sâu `0`).
- **Chiều cao** của nút = số cạnh của **đường dài nhất từ nút xuống một lá** trong cây con của nó (lá có chiều cao `0`).

Công thức đệ quy cho chiều cao: `height(None) = −1` và `height(node) = 1 + max(height(left), height(right))` (lá: `1 + max(−1, −1) = 0`). Ta sẽ dùng quy ước **`None` trả `0`** và **chiều cao tính từ `1`** (lá là `1`) cho gọn đệ quy: khi đó chỉ số đợt là `height − 1`. Hai quy ước tương đương; ta chọn quy ước giống lời giải mẫu: hàm trả "chiều cao + 1" và chỉ số đợt chính là `max(trái, phải)`.

In [6]:
def height(node):
    """Quy ước: None -> 0, lá -> 1 (tức chiều cao + 1)."""
    if node is None:
        return 0
    return 1 + max(height(node.left), height(node.right))


tree = build_tree([1, 2, 3, 4, 5])
print(height(tree.left.left), height(tree.left), height(tree))     # 1 2 3  (lá 4 -> 1, nút 2 -> 2, gốc 1 -> 3)
chain = build_tree([1, 2, None, 3])                                # chuỗi 1 -> 2 -> 3
print(height(chain), height(chain.left), height(chain.left.left)) # 3 2 1

1 2 3
3 2 1


### 3.6. List của list và quy tắc "mở đợt mới"

Kết quả là `ans`, một list các list: `ans[0]` là đợt lá đầu tiên, `ans[1]` đợt thứ hai... Khi một nút thuộc đợt `h` mà `ans` chưa có `ans[h]`, ta phải **tạo đợt mới** trước khi `append`. Nhờ các nút thuộc đợt `h` luôn **xuất hiện sau** khi đợt `h − 1` đã được tạo (một nút chỉ có chiều cao `h` nếu có con chiều cao `h − 1`), điều kiện `len(ans) == h` đủ để biết cần mở đợt mới.

In [7]:
ans = []
for h, val in [(0, 4), (0, 5), (1, 2), (0, 3), (2, 1)]:
    if len(ans) == h:
        ans.append([])           # chưa có đợt h -> tạo
    ans[h].append(val)
    print(f"h={h}, val={val}: ans = {ans}")

h=0, val=4: ans = [[4]]
h=0, val=5: ans = [[4, 5]]
h=1, val=2: ans = [[4, 5], [2]]
h=0, val=3: ans = [[4, 5, 3], [2]]
h=2, val=1: ans = [[4, 5, 3], [2], [1]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ `[1,2,3,4,5]`. Vẽ cây:

```
      1
     / \
    2   3
   / \
  4   5
```

**Đợt 1:** lá là `4, 5, 3` (các nút không có con). Xóa chúng ⇒ còn `1 → 2` (nút `2` không còn con). **Đợt 2:** lá là `2`. **Đợt 3:** còn `1`.

Bảng "nút nào thuộc đợt nào":

| nút | con trái | con phải | chiều cao (lá = 0) | đợt (đếm từ 1) |
|---|---|---|---|---|
| 4 | – | – | 0 | 1 |
| 5 | – | – | 0 | 1 |
| 3 | – | – | 0 | 1 |
| 2 | 4 | 5 | `1 + max(0, 0) = 1` | 2 |
| 1 | 2 | 3 | `1 + max(1, 0) = 2` | 3 |

**Quan sát:** nút `1` có hai con ở đợt `2` (nút `2`) và đợt `1` (nút `3`). Nó chỉ thành lá **khi cả hai con đã bị xóa**, tức sau đợt **muộn hơn** (đợt 2) ⇒ nó thuộc đợt `3`. Vậy **đợt của một nút = 1 + đợt muộn nhất của hai con** — chính là định nghĩa chiều cao.

In [8]:
def heights_table(root):
    """Trả dict nút.val -> chiều cao (lá = 0) cho cây có giá trị khác nhau (dùng để tính tay/in bảng)."""
    h = {}

    def dfs(node):
        if node is None:
            return -1
        hh = 1 + max(dfs(node.left), dfs(node.right))
        h[node.val] = hh
        return hh

    dfs(root)
    return h


print(heights_table(build_tree([1, 2, 3, 4, 5])))      # {4: 0, 5: 0, 2: 1, 3: 0, 1: 2}

{4: 0, 5: 0, 2: 1, 3: 0, 1: 2}


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng việc xóa lá từng đợt

**Ý tưởng.** Làm đúng như đề: mỗi lượt duyệt cả cây một lần, **ghi lại các lá và cắt chúng đi** (đặt con trỏ của cha về `None`). Lặp cho đến khi gốc cũng bị xóa. Hàm `strip(node, leaves)` trả về **gốc mới** của cây con sau khi xóa lá (`None` nếu `node` là lá).

Vì thao tác này **phá hủy cây**, ta làm việc trên **bản sao** (`clone_tree`) để không làm hỏng đầu vào dùng cho các cách khác.

In [9]:
def clone_tree(node):
    if node is None:
        return None
    return TreeNode(node.val, clone_tree(node.left), clone_tree(node.right))


def find_leaves_simulate(root):
    root = clone_tree(root)                  # không phá cây gốc của người gọi
    ans = []

    def strip(node, leaves):
        """Xóa lá trong cây con của node; trả về gốc mới của cây con."""
        if node is None:
            return None
        if node.left is None and node.right is None:
            leaves.append(node.val)          # node là lá -> thu thập
            return None                      # và "xóa": cha sẽ gán con trỏ về None
        node.left = strip(node.left, leaves)     # PHẢI gán lại để cắt liên kết
        node.right = strip(node.right, leaves)
        return node

    while root is not None:
        leaves = []
        root = strip(root, leaves)
        ans.append(leaves)
    return ans


for lst, expected in examples_lists:
    print(find_leaves_simulate(build_tree(lst)), "(kỳ vọng", expected, ")")

[[4, 5, 3], [2], [1]] (kỳ vọng [[4, 5, 3], [2], [1]] )
[[1]] (kỳ vọng [[1]] )


**Độ phức tạp (Cách 1).** Gọi `n` là số nút, `h` là chiều cao cây. Mỗi lượt duyệt `O(số nút còn lại)`, và có `h + 1` lượt ⇒ **Time `O(n·h)`**, tệ nhất cây "cái chổi" (chuỗi) là **`O(n²)`**. **Space `O(h)`** cho ngăn xếp đệ quy (và `O(n)` cho bản sao).

**Nút thắt:** ta duyệt lại gần như cả cây ở **mỗi** đợt, trong khi thực ra mỗi nút chỉ cần biết **nó bị xóa ở đợt nào**. Đợt đó phụ thuộc cấu trúc cây con (chiều cao) — tính được trong **một** lượt DFS.

## 6. Ý tưởng thuật toán / bước đột phá

**Đợt xóa của một nút = chiều cao của nó** (lá: `0`):

- Lá bị xóa ở đợt `0` (đợt đầu).
- Một nút có con trái ở đợt `a` và con phải ở đợt `b` sẽ trở thành lá ngay **sau khi** cả hai con bị xóa, tức bị xóa ở đợt `max(a, b) + 1` (nếu thiếu một con, coi con đó là "đã bị xóa từ đầu", đợt `−1`).

Do vậy trong **một lượt DFS hậu thứ tự**: tính `h` của hai con trước, đặt `h = max(h_trái, h_phải)` (đây là chỉ số đợt, đếm từ `0`), thêm `node.val` vào `ans[h]` (tạo đợt mới nếu `len(ans) == h`), rồi **trả về `h + 1`** cho cha. Quy ước: hàm trả "chiều cao + 1" (`None` trả `0`, lá trả `1`) nên chỉ số đợt của nút chính là `max(trả về của con trái, trả về của con phải)`.

Mỗi nút được thăm **đúng một lần** ⇒ `O(n)`.

In [10]:
def trace_heights(node, depth=0):
    """In từng bước: trả về 'chiều cao + 1' của mỗi nút, thụt lề theo độ sâu."""
    pad = "  " * depth
    if node is None:
        return 0
    print(f"{pad}vào {node.val}")
    l = trace_heights(node.left, depth + 1)
    r = trace_heights(node.right, depth + 1)
    h = max(l, r)
    print(f"{pad}ra  {node.val}: l={l}, r={r} -> đợt (chỉ số) = {h}, trả về {h + 1}")
    return h + 1


_ = trace_heights(build_tree([1, 2, 3, 4, 5]))

vào 1
  vào 2
    vào 4
    ra  4: l=0, r=0 -> đợt (chỉ số) = 0, trả về 1
    vào 5
    ra  5: l=0, r=0 -> đợt (chỉ số) = 0, trả về 1
  ra  2: l=1, r=1 -> đợt (chỉ số) = 1, trả về 2
  vào 3
  ra  3: l=0, r=0 -> đợt (chỉ số) = 0, trả về 1
ra  1: l=2, r=1 -> đợt (chỉ số) = 2, trả về 3


## 7. Các cách tiếp theo

### Cách 2 — DFS hậu thứ tự, bỏ nút vào `ans[h]` (lời giải chuẩn)

**Giải thích từng đoạn.**

1. `dfs(node)` trả về **chiều cao + 1** của cây con (`None` → `0`).
2. `l, r = dfs(node.left), dfs(node.right)`: xử lý hai con **trước** (postorder).
3. `h = max(l, r)`: chỉ số đợt của `node`.
4. `if len(ans) == h: ans.append([])`: mở đợt mới nếu cần; rồi `ans[h].append(node.val)`.
5. `return h + 1`: chiều cao + 1 của `node`, cho cha dùng.

In [11]:
def find_leaves_dfs(root):
    ans = []

    def dfs(node):
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        h = max(l, r)                  # đợt của node: sau khi cả hai con đã bị xóa
        if len(ans) == h:
            ans.append([])             # chưa có đợt h -> tạo
        ans[h].append(node.val)
        return h + 1

    dfs(root)
    return ans


for lst, expected in examples_lists:
    print(find_leaves_dfs(build_tree(lst)), "(kỳ vọng", expected, ")")

[[4, 5, 3], [2], [1]] (kỳ vọng [[4, 5, 3], [2], [1]] )
[[1]] (kỳ vọng [[1]] )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionDFS:
    def findLeaves(self, root: Optional[TreeNode]) -> List[List[int]]:
        def dfs(node: Optional[TreeNode]) -> int:
            if node is None:
                return 0
            l, r = dfs(node.left), dfs(node.right)
            h = max(l, r)
            if len(ans) == h:
                ans.append([])
            ans[h].append(node.val)
            return h + 1

        ans = []
        dfs(root)
        return ans


print(SolutionDFS().findLeaves(build_tree([1, 2, 3, 4, 5])))     # [[4, 5, 3], [2], [1]]

[[4, 5, 3], [2], [1]]


**Độ phức tạp Cách 2.** **Time `O(n)`** (mỗi nút thăm đúng một lần), **Space `O(h)`** cho ngăn xếp đệ quy (`h <= n`), cộng `O(n)` cho `ans`.

### Cách 3 — Bóc lớp bằng "đếm số con" (kiểu sắp xếp topo / Kahn)

**Giải thích.** Nhìn cây như đồ thị có hướng **con → cha**. Một nút "sẵn sàng bị xóa" khi **số con còn lại bằng `0`**:

1. Duyệt cây một lần (ngăn xếp), ghi `parent[node]` và `remaining[node]` = **số con** (`0`, `1` hoặc `2`). Các nút có `remaining == 0` là **lá**: đó là đợt đầu `current`.
2. Mỗi vòng: ghi các giá trị của `current` vào `ans`; với mỗi `n` trong `current`, **giảm `remaining[parent[n]]` đi `1`**; nếu về `0` thì cha thành lá, đưa vào đợt kế tiếp `nxt`.
3. Dừng khi `current` rỗng.

Ý tưởng khác Cách 2: không dùng chiều cao mà **bóc lớp** như **BFS từ các lá lên gốc** (thuật toán Kahn).

In [13]:
def find_leaves_peel(root):
    if root is None:
        return []
    parent = {root: None}
    remaining = {}                              # số con chưa bị xóa của mỗi nút
    current = []                                # đợt lá đầu tiên
    stack = [root]
    while stack:                                # duyệt tiền thứ tự, trái trước phải
        node = stack.pop()
        children = [c for c in (node.left, node.right) if c is not None]
        remaining[node] = len(children)
        if not children:
            current.append(node)                # lá
        for c in children:
            parent[c] = node
        stack.extend(reversed(children))        # đẩy phải trước để pop ra trái trước

    ans = []
    while current:
        ans.append([n.val for n in current])    # đợt hiện tại
        nxt = []
        for n in current:
            p = parent[n]
            if p is not None:
                remaining[p] -= 1
                if remaining[p] == 0:           # con cuối cùng của p vừa bị xóa -> p thành lá
                    nxt.append(p)
        current = nxt
    return ans


for lst, expected in examples_lists:
    print(find_leaves_peel(build_tree(lst)), "(kỳ vọng", expected, ")")

[[4, 5, 3], [2], [1]] (kỳ vọng [[4, 5, 3], [2], [1]] )
[[1]] (kỳ vọng [[1]] )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionPeel:
    def findLeaves(self, root: Optional[TreeNode]) -> List[List[int]]:
        if root is None:
            return []
        parent = {root: None}
        remaining = {}
        current = []
        stack = [root]
        while stack:
            node = stack.pop()
            children = [c for c in (node.left, node.right) if c is not None]
            remaining[node] = len(children)
            if not children:
                current.append(node)
            for c in children:
                parent[c] = node
            stack.extend(reversed(children))
        ans = []
        while current:
            ans.append([n.val for n in current])
            nxt = []
            for n in current:
                p = parent[n]
                if p is not None:
                    remaining[p] -= 1
                    if remaining[p] == 0:
                        nxt.append(p)
            current = nxt
        return ans


print(SolutionPeel().findLeaves(build_tree([1, 2, 3, 4, 5])))

[[4, 5, 3], [2], [1]]


**Độ phức tạp Cách 3.** **Time `O(n)`** (mỗi nút vào `current` đúng một lần, mỗi cạnh được xử lý một lần), **Space `O(n)`** (`parent`, `remaining`, các đợt). Không đệ quy trong phần bóc lớp; chỉ dùng ngăn xếp tường minh để duyệt ban đầu.

### Cách 4 — DFS hậu thứ tự bằng ngăn xếp tường minh (không đệ quy)

**Giải thích.** Cùng ý tưởng "đợt = chiều cao" như Cách 2 nhưng không đệ quy. Dùng ngăn xếp các cặp `(node, đã_xử_lý_con)`: lần **đầu** gặp một nút, ta đẩy lại nó với cờ `True`, rồi đẩy **phải** và **trái** (để trái được lấy ra trước). Khi nút xuất hiện **lần thứ hai** (cờ `True`) thì cả hai cây con đã xử lý xong: đọc `height[con]` từ một `dict` rồi tính chiều cao của nút này. Dùng khi cây có thể rất sâu (tránh `RecursionError`).

In [15]:
def find_leaves_iter(root):
    ans = []
    height = {}                                      # height[node] = chiều cao (lá = 0)
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if node is None:
            continue
        if not done:
            stack.append((node, True))               # quay lại nút này sau khi xong hai con
            stack.append((node.right, False))
            stack.append((node.left, False))         # lấy ra trước -> trái được xử lý trước
        else:
            lh = height[node.left] if node.left else -1
            rh = height[node.right] if node.right else -1
            h = 1 + max(lh, rh)
            height[node] = h
            if len(ans) == h:
                ans.append([])
            ans[h].append(node.val)
    return ans


for lst, expected in examples_lists:
    print(find_leaves_iter(build_tree(lst)), "(kỳ vọng", expected, ")")

[[4, 5, 3], [2], [1]] (kỳ vọng [[4, 5, 3], [2], [1]] )
[[1]] (kỳ vọng [[1]] )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionIter:
    def findLeaves(self, root: Optional[TreeNode]) -> List[List[int]]:
        ans = []
        height = {}
        stack = [(root, False)]
        while stack:
            node, done = stack.pop()
            if node is None:
                continue
            if not done:
                stack.append((node, True))
                stack.append((node.right, False))
                stack.append((node.left, False))
            else:
                lh = height[node.left] if node.left else -1
                rh = height[node.right] if node.right else -1
                h = 1 + max(lh, rh)
                height[node] = h
                if len(ans) == h:
                    ans.append([])
                ans[h].append(node.val)
        return ans


print(SolutionIter().findLeaves(build_tree([1, 2, 3, 4, 5])))

[[4, 5, 3], [2], [1]]


**Độ phức tạp Cách 4.** **Time `O(n)`**, **Space `O(n)`** (`height` và ngăn xếp). Không bị giới hạn đệ quy của Python.

(Chú ý: khóa của `height` là **đối tượng nút**, không phải `val` — nhiều nút có thể trùng giá trị!)

## 8. TRACE từng bước

Ta trace **Cách 2** trên một cây "lệch" để thấy các nút cùng đợt đến từ nhiều nhánh: `[1,2,3,4,5,6,None,7]`, tức cây `1(2(4(7),5), 3(6))`.

In [17]:
show_tree(build_tree([1, 2, 3, 4, 5, 6, None, 7]))

    3
        6
1
        5
    2
        4
            7


In [18]:
def trace_find_leaves(root):
    ans = []

    def dfs(node, depth=0):
        pad = "  " * depth
        if node is None:
            return 0
        print(f"{pad}vào {node.val}")
        l, r = dfs(node.left, depth + 1), dfs(node.right, depth + 1)
        h = max(l, r)
        opened = ""
        if len(ans) == h:
            ans.append([])
            opened = f" (mở đợt mới ans[{h}])"
        ans[h].append(node.val)
        print(f"{pad}ra  {node.val}: l={l}, r={r}, h={h}{opened} -> ans = {ans}, trả về {h + 1}")
        return h + 1

    dfs(root)
    return ans


result = trace_find_leaves(build_tree([1, 2, 3, 4, 5, 6, None, 7]))
print("kết quả:", result)
assert result == [[7, 5, 6], [4, 3], [2], [1]]

vào 1
  vào 2
    vào 4
      vào 7
      ra  7: l=0, r=0, h=0 (mở đợt mới ans[0]) -> ans = [[7]], trả về 1
    ra  4: l=1, r=0, h=1 (mở đợt mới ans[1]) -> ans = [[7], [4]], trả về 2
    vào 5
    ra  5: l=0, r=0, h=0 -> ans = [[7, 5], [4]], trả về 1
  ra  2: l=2, r=1, h=2 (mở đợt mới ans[2]) -> ans = [[7, 5], [4], [2]], trả về 3
  vào 3
    vào 6
    ra  6: l=0, r=0, h=0 -> ans = [[7, 5, 6], [4], [2]], trả về 1
  ra  3: l=1, r=0, h=1 -> ans = [[7, 5, 6], [4, 3], [2]], trả về 2
ra  1: l=3, r=2, h=3 (mở đợt mới ans[3]) -> ans = [[7, 5, 6], [4, 3], [2], [1]], trả về 4
kết quả: [[7, 5, 6], [4, 3], [2], [1]]


Bảng trace Cách 2 (theo thứ tự "ra" khỏi từng nút, tức hậu thứ tự):

| Thứ tự ra | nút | `l` | `r` | `h = max(l, r)` | thêm vào | `ans` sau bước |
|---|---|---|---|---|---|---|
| 1 | 7 | 0 | 0 | 0 | `ans[0]` (mở mới) | `[[7]]` |
| 2 | 4 | 1 | 0 | 1 | `ans[1]` (mở mới) | `[[7],[4]]` |
| 3 | 5 | 0 | 0 | 0 | `ans[0]` | `[[7,5],[4]]` |
| 4 | 2 | 2 | 1 | 2 | `ans[2]` (mở mới) | `[[7,5],[4],[2]]` |
| 5 | 6 | 0 | 0 | 0 | `ans[0]` | `[[7,5,6],[4],[2]]` |
| 6 | 3 | 1 | 0 | 1 | `ans[1]` | `[[7,5,6],[4,3],[2]]` |
| 7 | 1 | 3 | 2 | 3 | `ans[3]` (mở mới) | `[[7,5,6],[4,3],[2],[1]]` |

Lưu ý nút `2`: con trái `4` trả `2` (chiều cao `1` + 1), con phải `5` trả `1` ⇒ `h = max(2, 1) = 2`: nút `2` bị xóa **sau** nút `4` (đợt `1`) dù `5` đã bị xóa ở đợt `0`. Và nút `1` ở đợt `3` vì nhánh trái sâu hơn nhánh phải.

Giờ trace **Cách 3 (bóc lớp)** để thấy từng đợt và việc giảm `remaining` của cha:

In [19]:
def trace_peel(root):
    parent = {root: None}
    remaining = {}
    current = []
    stack = [root]
    while stack:
        node = stack.pop()
        children = [c for c in (node.left, node.right) if c is not None]
        remaining[node] = len(children)
        if not children:
            current.append(node)
        for c in children:
            parent[c] = node
        stack.extend(reversed(children))
    print("số con ban đầu:", {n.val: remaining[n] for n in remaining})
    ans, rnd = [], 1
    while current:
        vals = [n.val for n in current]
        ans.append(vals)
        print(f"đợt {rnd}: lá = {vals}")
        nxt = []
        for n in current:
            p = parent[n]
            if p is not None:
                remaining[p] -= 1
                note = " -> thành lá!" if remaining[p] == 0 else ""
                print(f"    xóa {n.val}: cha {p.val} còn {remaining[p]} con{note}")
                if remaining[p] == 0:
                    nxt.append(p)
        current, rnd = nxt, rnd + 1
    return ans


assert trace_peel(build_tree([1, 2, 3, 4, 5, 6, None, 7])) == [[7, 5, 6], [4, 3], [2], [1]]

số con ban đầu: {1: 2, 2: 2, 4: 1, 7: 0, 5: 0, 3: 1, 6: 0}
đợt 1: lá = [7, 5, 6]
    xóa 7: cha 4 còn 0 con -> thành lá!
    xóa 5: cha 2 còn 1 con
    xóa 6: cha 3 còn 0 con -> thành lá!
đợt 2: lá = [4, 3]
    xóa 4: cha 2 còn 0 con -> thành lá!
    xóa 3: cha 1 còn 1 con
đợt 3: lá = [2]
    xóa 2: cha 1 còn 0 con -> thành lá!
đợt 4: lá = [1]


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2** (DFS hậu thứ tự): ngắn, `O(n)`.

In [20]:
from typing import List, Optional


class Solution:
    def findLeaves(self, root: Optional[TreeNode]) -> List[List[int]]:
        def dfs(node: Optional[TreeNode]) -> int:
            if node is None:
                return 0                       # None: chiều cao + 1 = 0
            l, r = dfs(node.left), dfs(node.right)
            h = max(l, r)                      # đợt bị xóa của node (đếm từ 0)
            if len(ans) == h:
                ans.append([])                 # mở đợt mới
            ans[h].append(node.val)
            return h + 1

        ans = []
        dfs(root)
        return ans


sol = Solution()
for lst, expected in examples_lists:
    print(lst, "->", sol.findLeaves(build_tree(lst)))
    assert sol.findLeaves(build_tree(lst)) == expected

[1, 2, 3, 4, 5] -> [[4, 5, 3], [2], [1]]
[1] -> [[1]]


## 10. Vì sao thuật toán đúng

Định nghĩa `r(v)` là **đợt (đếm từ `0`) mà nút `v` bị xóa** theo mô tả của đề.

**Khẳng định:** `r(v) = 1 + max(r(trái), r(phải))`, với `r(None) = −1`.

- **Lá:** `v` không có con ⇒ là lá ngay đầu ⇒ bị xóa ở đợt `0`; công thức cho `1 + max(−1, −1) = 0` ✓.
- **Nút có con:** `v` **chưa là lá** chừng nào còn con. Con `c` bị xóa ở đợt `r(c)`, và xóa **xong** ở cuối đợt đó, nên `v` **là lá** ở đầu đợt `max(r(trái), r(phải)) + 1` (mọi con đã xóa) — và không sớm hơn. Mọi lá ở một đợt được thu thập đúng ở đợt đó (đề thu **tất cả** lá hiện có). Do đó `r(v) = 1 + max(...)` ✓. Nếu chỉ có một con, con còn lại coi như đã xóa (`−1`) ✓.

**DFS hậu thứ tự** tính chính xác `r` cho mọi nút: khi ra khỏi `v` ta đã có giá trị của hai con (`dfs` trả `r + 1`, `None` trả `0 = (−1) + 1`), `h = max(l, r)` chính là `r(v)`, và `dfs` trả `h + 1 = r(v) + 1` ✓.

**Mở đợt mới đúng lúc:** `ans[h]` chỉ bị truy cập khi `len(ans) >= h`; vì nút có `r(v) = h > 0` có một con `c` với `r(c) = h − 1`, và `c` được xử lý **trước** `v`, nên `ans[h − 1]` đã tồn tại ⇒ `len(ans) >= h`. Khi `len(ans) == h` ta mở đúng đợt tiếp theo. (Với `h = 0`: `ans` có thể rỗng, `len(ans) == 0 == h` ⇒ mở.)

**Mỗi nút đúng một đợt:** mỗi nút được `append` **đúng một lần** (thăm đúng một lần), vào `ans[r(v)]` ✓.

**Cách 3 (bóc lớp)** cũng đúng theo định nghĩa: `remaining[v]` đếm số con **chưa bị xóa**; `v` vào đợt kế khi nó về `0` — tức vừa sau khi con cuối bị xóa, đúng như "trở thành lá".

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao cây.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng xóa lá từng đợt | `O(n·h)`, tệ nhất `O(n²)` | `O(n)` (bản sao + đệ quy) |
| 2. DFS hậu thứ tự theo chiều cao | `O(n)` | `O(h)` + `O(n)` cho `ans` |
| 3. Bóc lớp bằng đếm số con | `O(n)` | `O(n)` |
| 4. DFS hậu thứ tự bằng ngăn xếp | `O(n)` | `O(n)` |

Với `n <= 100` mọi cách đều tức thì; sự khác biệt cho thấy khi cây lớn (hàng chục nghìn nút) Cách 1 sẽ chậm.

## 12. Bộ test

Vì **thứ tự trong mỗi đợt không quan trọng**, ta so sánh **sau khi sắp xếp từng đợt** (`normalize`). Riêng ví dụ 1, ta kiểm tra thêm kết quả **đúng thứ tự** của các cách (trái → phải) khớp chính xác với đề.

In [21]:
sol = Solution()
impls = {
    "simulate": find_leaves_simulate,
    "dfs": find_leaves_dfs,
    "peel": find_leaves_peel,
    "iter": find_leaves_iter,
    "lc_dfs": SolutionDFS().findLeaves,
    "lc_peel": SolutionPeel().findLeaves,
    "lc_iter": SolutionIter().findLeaves,
    "leetcode": sol.findLeaves,
}


def normalize(levels):
    return [sorted(level) for level in levels]


def check_all(values, expected, label):
    for key, fn in impls.items():
        actual = fn(build_tree(values))
        assert normalize(actual) == normalize(expected), (label, key, actual, expected)
    print("PASS", label)


for idx, (lst, expected) in enumerate(examples_lists, 1):
    check_all(lst, expected, f"ví dụ {idx}")
    # ví dụ của đề: thứ tự trái -> phải khớp chính xác
    for key in ("dfs", "iter", "simulate", "leetcode"):
        assert impls[key](build_tree(lst)) == expected, key

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Một nút; chuỗi lệch trái; chuỗi lệch phải; cây đầy đủ; cây zíc-zắc; giá trị âm/trùng nhau.

In [22]:
check_all([1], [[1]], "một nút")
check_all([1, 2], [[2], [1]], "gốc chỉ có con trái")
check_all([1, None, 2], [[2], [1]], "gốc chỉ có con phải")
check_all([1, 2, None, 3, None, 4], [[4], [3], [2], [1]], "chuỗi lệch trái 4 nút")
check_all([1, None, 2, None, 3, None, 4], [[4], [3], [2], [1]], "chuỗi lệch phải 4 nút")
check_all([1, 2, 3], [[2, 3], [1]], "gốc với hai lá")
check_all([1, 2, 3, 4, 5, 6, 7], [[4, 5, 6, 7], [2, 3], [1]], "cây đầy đủ 3 tầng")
check_all([1, 2, 3, None, 4, 5], [[4, 5], [2, 3], [1]], "con trái chỉ có con phải")
check_all([1, 2, 3, 4, 5, 6, None, 7], [[7, 5, 6], [4, 3], [2], [1]], "cây lệch (ví dụ mục 8)")
check_all([0, 0, 0, 0, 0], [[0, 0, 0], [0], [0]], "toàn số 0 (giá trị trùng)")
check_all([-100, 100, -100], [[-100, 100], [-100]], "giá trị biên -100, 100")
check_all([5, 5, 5, None, 5, 5], [[5, 5], [5, 5], [5]], "giá trị trùng ở nhiều đợt")

# cây zíc-zắc 10 nút: mỗi nút một con xen kẽ trái/phải
vals, cur = [1], 1
for k in range(9):
    vals += [None, k + 2] if k % 2 else [k + 2, None]
print(vals)
check_all(vals, [[10], [9], [8], [7], [6], [5], [4], [3], [2], [1]], "zíc-zắc 10 nút")

PASS một nút
PASS gốc chỉ có con trái
PASS gốc chỉ có con phải
PASS chuỗi lệch trái 4 nút
PASS chuỗi lệch phải 4 nút
PASS gốc với hai lá
PASS cây đầy đủ 3 tầng
PASS con trái chỉ có con phải
PASS cây lệch (ví dụ mục 8)
PASS toàn số 0 (giá trị trùng)
PASS giá trị biên -100, 100
PASS giá trị trùng ở nhiều đợt
[1, 2, None, None, 3, 4, None, None, 5, 6, None, None, 7, 8, None, None, 9, 10, None]
PASS zíc-zắc 10 nút


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi hay gặp: (a) **dùng `min`** thay `max` để chọn đợt; (b) **đảo thứ tự duyệt theo tầng** (BFS từ gốc rồi đảo) — nhầm **độ sâu** với **chiều cao**.

In [23]:
def wrong_min(root):
    ans = []

    def dfs(node):
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        h = min(l, r)                               # SAI: phải là max
        if len(ans) == h:
            ans.append([])
        ans[h].append(node.val)
        return h + 1

    dfs(root)
    return ans


def wrong_reverse_levels(root):
    """SAI: nhầm độ sâu với chiều cao: BFS theo tầng từ gốc rồi đảo."""
    if root is None:
        return []
    levels, q = [], deque([root])
    while q:
        levels.append([n.val for n in q])
        q = deque(c for n in q for c in (n.left, n.right) if c)
    return levels[::-1]


tree = build_tree([1, 2, 3, 4, 5])
print("min:", wrong_min(tree), "!=", find_leaves_dfs(tree))
print("đảo BFS:", wrong_reverse_levels(tree), "!=", find_leaves_dfs(tree))
assert normalize(wrong_min(tree)) != normalize(find_leaves_dfs(tree))
assert normalize(wrong_reverse_levels(tree)) != normalize(find_leaves_dfs(tree))

min: [[4, 5, 3], [2, 1]] != [[4, 5, 3], [2], [1]]
đảo BFS: [[4, 5], [2, 3], [1]] != [[4, 5, 3], [2], [1]]


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh cây nhị phân **đúng ràng buộc** (`1..100` nút, giá trị trong `[-100, 100]`, có trùng giá trị) theo nhiều dạng: ngẫu nhiên, chuỗi lệch, cân bằng. Oracle: **mô phỏng xóa lá** (Cách 1), độc lập với tính chiều cao. Ngoài ra kiểm tra tính chất: tổng số giá trị bằng số nút, và đa tập giá trị khớp với cây.

In [24]:
def random_tree_values(rng, n, shape):
    """Sinh list kiểu LeetCode cho cây n nút bằng cách gắn dần nút mới vào chỗ trống ngẫu nhiên."""
    nodes = [TreeNode(rng.randint(-100, 100))]
    root = nodes[0]
    free = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        if shape == "left_chain":
            parent, side = nodes[-1], "left"
        elif shape == "right_chain":
            parent, side = nodes[-1], "right"
        elif shape == "balanced":
            parent, side = free.pop(0)
        else:
            parent, side = free.pop(rng.randrange(len(free)))
        child = TreeNode(rng.randint(-100, 100))
        setattr(parent, side, child)
        nodes.append(child)
        free += [(child, "left"), (child, "right")]
    return tree_to_list(root)


def all_values(root):
    return sorted(postorder_values(root))


rng = random.Random(366)
for trial in range(2000):
    n = rng.randint(1, 100)
    shape = rng.choice(["random", "random", "balanced", "left_chain", "right_chain"])
    values = random_tree_values(rng, n, shape)
    root = build_tree(values)
    ref = normalize(find_leaves_simulate(root))
    for key, fn in impls.items():
        got = fn(build_tree(values))
        assert normalize(got) == ref, (key, values)
    flat = sorted(v for level in find_leaves_dfs(root) for v in level)
    assert flat == all_values(root) and len(flat) == n
    assert len(find_leaves_dfs(root)) == height(root)             # số đợt = chiều cao (quy ước lá = 1)
print("PASS 2000 cây ngẫu nhiên (1..100 nút) đối chiếu với mô phỏng xóa lá")

PASS 2000 cây ngẫu nhiên (1..100 nút) đối chiếu với mô phỏng xóa lá


### 12.4. Cây không bị phá hủy

Các cách 2–4 **không được thay đổi** cây đầu vào (Cách 1 làm việc trên bản sao). Ta kiểm tra bằng cách so `tree_to_list` trước và sau.

In [25]:
values = [3, 9, 20, None, None, 15, 7, 1, 2]
before = tree_to_list(build_tree(values))
for key, fn in impls.items():
    root = build_tree(values)
    fn(root)
    assert tree_to_list(root) == before, key
print("PASS: không cách nào làm thay đổi cây đầu vào:", before)

PASS: không cách nào làm thay đổi cây đầu vào: [3, 9, 20, None, None, 15, 7, 1, 2]


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `min` thay `max` (nút phải đợi **cả hai** con)

In [26]:
tree = build_tree([1, 2, 3, 4, 5])
print("sai (min):", wrong_min(tree), "| đúng:", Solution().findLeaves(tree))

sai (min): [[4, 5, 3], [2, 1]] | đúng: [[4, 5, 3], [2], [1]]


**Vì sao sai:** nút `1` có con trái `2` (đợt `1`) và con phải `3` (đợt `0`). Nó chỉ thành lá khi **cả hai** con đã bị xóa, tức phải đợi con **muộn hơn** ⇒ `max`. Dùng `min` xếp `1` vào đợt sớm (khi `2` vẫn còn), mâu thuẫn đề. **Cách sửa:** `h = max(l, r)`.

### Lỗi 2: Nhầm độ sâu với chiều cao (đảo BFS)

In [27]:
print("sai (đảo BFS):", wrong_reverse_levels(tree), "| đúng:", Solution().findLeaves(tree))

sai (đảo BFS): [[4, 5], [2, 3], [1]] | đúng: [[4, 5, 3], [2], [1]]


**Vì sao sai:** BFS từ gốc gom theo **độ sâu**: `[[1], [2, 3], [4, 5]]`, đảo lại cho `[[4, 5], [2, 3], [1]]`. Nhưng nút `3` là lá ở **đợt đầu** dù có độ sâu `1`. Thứ quyết định đợt là **chiều cao** (khoảng cách tới lá sâu nhất bên dưới), không phải khoảng cách từ gốc. **Cách sửa:** tính chiều cao bằng DFS hậu thứ tự.

### Lỗi 3: Quên mở đợt mới → `IndexError`

In [28]:
def sai_quen_mo_dot(root):
    ans = []

    def dfs(node):
        if node is None:
            return 0
        l, r = dfs(node.left), dfs(node.right)
        h = max(l, r)
        ans[h].append(node.val)                     # SAI: ans[h] có thể chưa tồn tại
        return h + 1

    dfs(root)
    return ans


try:
    sai_quen_mo_dot(build_tree([1, 2, 3]))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** lúc đầu `ans` rỗng nên `ans[0]` chưa có. **Cách sửa:** `if len(ans) == h: ans.append([])` trước khi `append`. (Hoặc khởi tạo `defaultdict(list)` rồi đọc theo khóa tăng dần.)

### Lỗi 4: Mô phỏng nhưng quên cắt liên kết cha–con

In [29]:
def sai_simulate_khong_cat(root, max_rounds=6):
    root = clone_tree(root)
    ans = []

    def strip(node, leaves):
        if node is None:
            return None
        if node.left is None and node.right is None:
            leaves.append(node.val)
            return None
        strip(node.left, leaves)                    # SAI: không gán lại node.left = ...
        strip(node.right, leaves)
        return node

    rounds = 0
    while root is not None and rounds < max_rounds:      # max_rounds chỉ để không treo máy khi minh họa
        leaves = []
        root = strip(root, leaves)
        ans.append(leaves)
        rounds += 1
    return ans


print("sai:", sai_simulate_khong_cat(build_tree([1, 2, 3])))
print("đúng:", find_leaves_simulate(build_tree([1, 2, 3])))

sai: [[2, 3], [2, 3], [2, 3], [2, 3], [2, 3], [2, 3]]
đúng: [[2, 3], [1]]


**Vì sao sai:** hàm trả `None` cho lá, nhưng **cha không gán lại** `node.left`/`node.right`, nên lá vẫn còn trong cây; mỗi lượt lại thu thập lại **cùng** các lá ⇒ vòng `while` không bao giờ dừng (ở đây ta phải đặt `max_rounds`). **Cách sửa:** `node.left = strip(node.left, leaves)` và tương tự cho `right`.

### Lỗi 5: Dùng `node.val` làm khóa cho chiều cao khi giá trị trùng

In [30]:
def sai_khoa_theo_val(root):
    ans, height = [], {}
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if node is None:
            continue
        if not done:
            stack.append((node, True)); stack.append((node.right, False)); stack.append((node.left, False))
        else:
            lh = height[node.left.val] if node.left else -1      # SAI: khóa theo val (trùng giá trị -> lẫn lộn)
            rh = height[node.right.val] if node.right else -1
            h = 1 + max(lh, rh)
            height[node.val] = h
            if len(ans) == h:
                ans.append([])
            ans[h].append(node.val)
    return ans


dup = build_tree([1, 2, 2, 1])                 # cây 1(2(1), 2): có hai nút giá trị 1 và hai nút giá trị 2
print("sai:", sai_khoa_theo_val(dup), "| đúng:", find_leaves_iter(dup))
assert sai_khoa_theo_val(dup) != find_leaves_iter(dup)

sai: [[1, 2], [2, 1]] | đúng: [[1, 2], [2], [1]]


**Vì sao sai:** trong cây `1(2(1), 2)`, nút `2` bên trái (chiều cao `1`) và nút `2` bên phải (lá, chiều cao `0`) có cùng khóa `2`; nút xử lý sau (lá phải, `0`) **ghi đè** nút trước. Gốc `1` tra `height[2] = 0` thay vì `1` nên tính ra `h = 1` và bị xếp nhầm vào cùng đợt với nút `2` bên trái (bản sai ra `[[1, 2], [2, 1]]`, bản đúng `[[1, 2], [2], [1]]`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Xóa lá lặp lại / bóc từng lớp từ lá lên**" ⇒ **chiều cao** của nút (khoảng cách tới lá xa nhất bên dưới); dùng **DFS hậu thứ tự** trả về chiều cao.
- Quy tắc **"nút chờ cả hai con"** ⇒ `max` (còn "đường ngắn nhất tới lá" mới là `min`).
- Cây là đồ thị: có thể **bóc lớp theo bậc** (đếm số con còn lại) giống **Kahn** — hữu ích khi đề cho đồ thị tổng quát.
- Kết quả "danh sách các đợt" ⇒ `ans[h].append(...)` với `if len(ans) == h: ans.append([])`.
- Trong DFS hậu thứ tự: **con trước, cha sau**; cái cha cần từ con thường được **trả về** bằng `return`.
- Khóa `dict` nên là **đối tượng nút** (không phải `val`) khi giá trị có thể trùng.
- Luôn thử các hình dạng cực đoan: **một nút**, **chuỗi lệch trái/phải**, **cây đầy đủ**.

## 15. Biến thể của bài

1. **Delete Leaves With a Given Value (1325):** xóa lặp lại các lá có giá trị `target` (lá mới xuất hiện sau khi xóa cũng bị xét).
2. **Maximum Depth of Binary Tree (104)** và **Diameter of Binary Tree (543):** cùng khung "DFS hậu thứ tự trả về chiều cao".
3. **Trả về chiều cao của mọi nút** (một `dict` đối tượng → chiều cao).
4. **Thu thập theo độ sâu** (BFS thường: Binary Tree Level Order Traversal 102) — phân biệt với bài này.
5. **Các lá ở "đợt" từ trên xuống** (đảo thứ tự các đợt của kết quả).
6. **N-ary tree** (cây nhiều con): `h = 1 + max(h các con)`.

In [31]:
def remove_leaves_with_value(root, target):
    """LeetCode 1325: xóa lá có val == target, lặp lại."""
    if root is None:
        return None
    root.left = remove_leaves_with_value(root.left, target)
    root.right = remove_leaves_with_value(root.right, target)
    if root.left is None and root.right is None and root.val == target:
        return None
    return root


t = build_tree([1, 2, 3, 2, None, 2, 4])
print(tree_to_list(remove_leaves_with_value(t, 2)))              # [1, None, 3, None, 4]
t = build_tree([1, 3, 3, 3, 2])
print(tree_to_list(remove_leaves_with_value(t, 3)))              # [1, 3, None, None, 2]
assert tree_to_list(remove_leaves_with_value(build_tree([1, 2, 3, 2, None, 2, 4]), 2)) == [1, None, 3, None, 4]
assert tree_to_list(remove_leaves_with_value(build_tree([1, 3, 3, 3, 2]), 3)) == [1, 3, None, None, 2]


def heights_of_all(root):
    """dict nút -> chiều cao (lá = 0)."""
    h = {}

    def dfs(node):
        if node is None:
            return -1
        h[node] = 1 + max(dfs(node.left), dfs(node.right))
        return h[node]

    dfs(root)
    return h


root = build_tree([1, 2, 3, 4, 5])
hs = heights_of_all(root)
print(sorted((n.val, hh) for n, hh in hs.items()))              # [(1, 2), (2, 1), (3, 0), (4, 0), (5, 0)]
print(find_leaves_dfs(root)[::-1])                              # đảo thứ tự đợt: [[1], [2], [4, 5, 3]]

[1, None, 3, None, 4]
[1, 3, None, None, 2]
[(1, 2), (2, 1), (3, 0), (4, 0), (5, 0)]
[[1], [2], [4, 5, 3]]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng xóa lá | mỗi lượt cắt các lá thật sự | `O(n·h)` (tệ nhất `O(n²)`) | `O(n)` | hiểu đề, oracle |
| 2. DFS hậu thứ tự | đợt = chiều cao = `max(con) + 1` | `O(n)` | `O(h)` + `O(n)` | **nên nộp** |
| 3. Bóc lớp theo số con | Kahn: lá → cha khi hết con | `O(n)` | `O(n)` | tư duy đồ thị / topo |
| 4. Hậu thứ tự bằng ngăn xếp | như Cách 2, không đệ quy | `O(n)` | `O(n)` | cây rất sâu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `build_tree([1, None, 2]).right.val`; (b) `tree_to_list(build_tree([1, 2, 3, None, 4]))`; (c) `Solution().findLeaves(build_tree([1]))`; (d) `Solution().findLeaves(build_tree([1, 2, None, 3]))`; (e) `height(build_tree([1, 2, 3]))` với quy ước `None → 0`, lá `→ 1`.

**Bài 2 (tính tay).** Với cây `[1,2,3,4,5,6,None,7]` (đã vẽ ở mục 8) hãy tính **chiều cao** của từng nút (lá = 0) và viết `findLeaves` theo thứ tự trái → phải; làm lại với cây `[1,2,3,4,None,None,5,6]` (hãy tự vẽ cây trước).

**Bài 3 (điền TODO).** Hoàn thành `find_leaves_todo(root)` theo Cách 2.

In [32]:
def find_leaves_todo(root):
    ans = []

    def dfs(node):
        # TODO 1: nếu node là None thì trả về 0
        # TODO 2: l = dfs(node.left); r = dfs(node.right)
        # TODO 3: h = max(l, r)
        # TODO 4: nếu len(ans) == h thì thêm một list rỗng vào ans
        # TODO 5: ans[h].append(node.val); trả về h + 1
        pass

    # TODO 6: gọi dfs(root) rồi trả về ans
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `find_leaves_levels_by_height(root)` theo cách **hai bước**: (1) một DFS tính `height[node]` (dict theo **đối tượng nút**) cho mọi nút; (2) một lượt duyệt tiền thứ tự (hoặc DFS bất kỳ) để gom `node.val` vào `ans[height[node]]`. Không dùng cách "mở đợt khi `len(ans) == h`" mà dùng `ans = [[] for _ in range(height[root] + 1)]`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_rounds(root)` trả **số đợt** (không dựng `ans`), và kiểm tra bằng `len(findLeaves(root))`. (b) **Delete Leaves With a Given Value (1325):** viết `delete_leaves(root, target)`, kiểm tra `[1,2,3,2,None,2,4], target=2 → [1,None,3,None,4]` và `[1,3,3,3,2], target=3 → [1,3,None,None,2]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`; `[1, 2, 3, None, 4]`; `[[1]]`; `[[3], [2], [1]]` (chuỗi lệch trái); `2`. Bài 3: nhớ `return 0` cho `None` và `return h + 1` ở cuối.
- **Mức 2 (vừa):** Bài 2: chiều cao: `7:0, 4:1, 5:0, 2:2, 6:0, 3:1, 1:3`; thứ tự gom theo hậu thứ tự trái → phải cho `[[7,5,6],[4,3],[2],[1]]`. Bài 4: `height[node] = 1 + max(...)` với `None → -1`.
- **Mức 3 (sâu):** Bài 5(a): `dfs` trả `1 + max(l, r)` rồi số đợt là `dfs(root)` (quy ước `None → 0`). 5(b): DFS **hậu thứ tự**: gán `root.left = delete_leaves(root.left, target)`, `root.right = ...`, rồi nếu `root` là lá và `val == target` trả `None`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert build_tree([1, None, 2]).right.val == 2
assert tree_to_list(build_tree([1, 2, 3, None, 4])) == [1, 2, 3, None, 4]
assert Solution().findLeaves(build_tree([1])) == [[1]]
assert Solution().findLeaves(build_tree([1, 2, None, 3])) == [[3], [2], [1]]
assert height(build_tree([1, 2, 3])) == 2

# Bài 2
r2 = build_tree([1, 2, 3, 4, 5, 6, None, 7])
assert sorted((n.val, hh) for n, hh in heights_of_all(r2).items()) == [(1, 3), (2, 2), (3, 1), (4, 1), (5, 0), (6, 0), (7, 0)]
assert Solution().findLeaves(r2) == [[7, 5, 6], [4, 3], [2], [1]]
# [1,2,3,4,None,None,5,6]: 1(2(4(6)),3(None,5)) -> lá 6, 5; rồi 4, 3; rồi 2; rồi 1
r2b = build_tree([1, 2, 3, 4, None, None, 5, 6])
assert Solution().findLeaves(r2b) == [[6, 5], [4, 3], [2], [1]]


# Bài 3
def find_leaves_todo_ans(root):
    ans = []

    def dfs(node):
        if node is None:
            return 0
        l = dfs(node.left)
        r = dfs(node.right)
        h = max(l, r)
        if len(ans) == h:
            ans.append([])
        ans[h].append(node.val)
        return h + 1

    dfs(root)
    return ans


for lst, expected in examples_lists:
    assert find_leaves_todo_ans(build_tree(lst)) == expected


# Bài 4
def find_leaves_levels_by_height(root):
    height = {}

    def dfs(node):
        if node is None:
            return -1
        height[node] = 1 + max(dfs(node.left), dfs(node.right))
        return height[node]

    dfs(root)
    ans = [[] for _ in range(height[root] + 1)]

    def collect(node):                      # tiền thứ tự: nút, trái, phải
        if node is None:
            return
        ans[height[node]].append(node.val)
        collect(node.left)
        collect(node.right)

    collect(root)
    return ans


for lst, expected in examples_lists:
    assert normalize(find_leaves_levels_by_height(build_tree(lst))) == normalize(expected)
rng = random.Random(3660)
for _ in range(300):
    vals = random_tree_values(rng, rng.randint(1, 60), rng.choice(["random", "balanced", "left_chain"]))
    assert normalize(find_leaves_levels_by_height(build_tree(vals))) == normalize(find_leaves_simulate(build_tree(vals)))


# Bài 5
def count_rounds(root):
    def dfs(node):
        if node is None:
            return 0
        return 1 + max(dfs(node.left), dfs(node.right))
    return dfs(root)


for _ in range(300):
    vals = random_tree_values(rng, rng.randint(1, 60), rng.choice(["random", "balanced", "right_chain"]))
    assert count_rounds(build_tree(vals)) == len(Solution().findLeaves(build_tree(vals)))


def delete_leaves(root, target):
    if root is None:
        return None
    root.left = delete_leaves(root.left, target)
    root.right = delete_leaves(root.right, target)
    if root.left is None and root.right is None and root.val == target:
        return None
    return root


assert tree_to_list(delete_leaves(build_tree([1, 2, 3, 2, None, 2, 4]), 2)) == [1, None, 3, None, 4]
assert tree_to_list(delete_leaves(build_tree([1, 3, 3, 3, 2]), 3)) == [1, 3, None, None, 2]
assert tree_to_list(delete_leaves(build_tree([1, 2, 2, 2, 2]), 2)) == [1]
print("Tất cả đáp án đều PASS")

Tất cả đáp án đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DFS hậu thứ tự, đợt = chiều cao)

```python
class Solution:
    def findLeaves(self, root: Optional[TreeNode]) -> List[List[int]]:
        def dfs(node):
            if node is None:
                return 0
            l, r = dfs(node.left), dfs(node.right)
            h = max(l, r)
            if len(ans) == h:
                ans.append([])
            ans[h].append(node.val)
            return h + 1

        ans = []
        dfs(root)
        return ans
```

**Checklist**

- [ ] Đợt của một nút là **chiều cao** của nó (không phải độ sâu): `h = max(l, r)` — dùng `max`, không phải `min`?
- [ ] `None` trả `0`, và hàm trả `h + 1` (chiều cao + 1) cho cha?
- [ ] Có `if len(ans) == h: ans.append([])` **trước** `ans[h].append(...)`?
- [ ] Hậu thứ tự: xử lý **hai con trước**, rồi mới thêm nút hiện tại?
- [ ] Nếu mô phỏng: nhớ **gán lại** `node.left = ...`/`node.right = ...` để thực sự cắt lá; và không phá cây đầu vào khi cần dùng lại?
- [ ] Đã thử cây một nút, chuỗi lệch trái/phải, cây đầy đủ, giá trị trùng; thứ tự trong mỗi đợt không quan trọng nên so sánh sau khi sắp xếp?
- [ ] Không khai báo lại `TreeNode` trong bài nộp (LeetCode đã có)?